# Entrega 2 — Detecção de Duplicatas Semânticas
**Ouvidoria Inteligente: Triagem Semântica de Manifestações Cidadãs**

Implementação da função `detectar_duplicatas`, matriz de similaridade completa e análise de falsos positivos/negativos frente ao gabarito conhecido do dataset.

## 1. Carregamento do corpus e dos embeddings

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

with open("manifestacoes.json", encoding="utf-8") as f:
    manifestacoes = json.load(f)

df = pd.DataFrame(manifestacoes)
textos = df["texto"].tolist()
ids = df["id"].tolist()

modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = modelo.encode(textos, show_progress_bar=True, normalize_embeddings=True)
print(f"Embeddings: {embeddings.shape}")

## 2. Função `detectar_duplicatas`
Recebe a lista de textos (e opcionalmente embeddings já calculados, para evitar recomputar o modelo) e retorna todos os pares com similaridade de cosseno igual ou acima do limiar.

In [ ]:
def detectar_duplicatas(textos, limiar=0.85, ids=None, embeddings=None, modelo=None):
    """
    Detecta pares de manifestações semanticamente duplicadas.

    Parâmetros
    ----------
    textos : list[str]
        Lista de textos das manifestações.
    limiar : float
        Limiar de similaridade de cosseno (0 a 1) acima do qual dois
        textos são considerados duplicatas semânticas. Padrão: 0.85.
    ids : list[str], opcional
        Identificadores das manifestações (na mesma ordem de `textos`).
        Se omitido, usa o índice posicional.
    embeddings : np.ndarray, opcional
        Embeddings pré-calculados. Se omitido, são calculados com `modelo`.
    modelo : SentenceTransformer, opcional
        Necessário apenas se `embeddings` não for fornecido.

    Retorna
    -------
    pares : list[dict]
        Lista de pares duplicados, ordenada por similaridade decrescente.
    matriz_similaridade : np.ndarray
        Matriz de similaridade de cosseno completa (n x n).
    """
    if embeddings is None:
        if modelo is None:
            raise ValueError("Forneça 'embeddings' já calculados ou um 'modelo'.")
        embeddings = modelo.encode(textos, normalize_embeddings=True)

    n = len(textos)
    ids = ids if ids is not None else list(range(n))
    matriz_similaridade = cosine_similarity(embeddings)

    pares = []
    for i in range(n):
        for j in range(i + 1, n):
            sim = float(matriz_similaridade[i, j])
            if sim >= limiar:
                pares.append({
                    "id_1": ids[i], "id_2": ids[j],
                    "texto_1": textos[i], "texto_2": textos[j],
                    "similaridade": round(sim, 4),
                })

    pares.sort(key=lambda p: -p["similaridade"])
    return pares, matriz_similaridade

## 3. Justificativa do limiar (0.85)

O limiar de **0.85** foi adotado como padrão porque, empiricamente, pares de frases que descrevem o mesmo fato concreto (mesmo endereço, mesmo tipo de problema, mesmo órgão envolvido) tendem a ficar acima dessa marca com modelos multilíngues de sentence-embeddings, enquanto textos apenas do mesmo *tema* (ex.: duas reclamações de saúde diferentes) costumam ficar na faixa de 0.5–0.7. Um limiar mais baixo (ex.: 0.6) aumentaria a cobertura, mas classificaria manifestações apenas relacionadas por tema (não duplicadas) como duplicatas (falsos positivos). Um limiar mais alto (ex.: 0.95) reduziria os falsos positivos, mas deixaria passar duplicatas reais escritas com vocabulário bem diferente (falsos negativos). 0.85 busca um equilíbrio conservador, priorizando precisão sobre cobertura — mais adequado para um sistema que vai agrupar manifestações automaticamente sem revisão humana prévia.

In [ ]:
LIMIAR = 0.85
pares_encontrados, matriz_sim = detectar_duplicatas(
    textos, limiar=LIMIAR, ids=ids, embeddings=embeddings
)

print(f"{len(pares_encontrados)} par(es) de duplicatas encontrados com limiar {LIMIAR}:\n")
for p in pares_encontrados:
    print(f"  {p['id_1']} × {p['id_2']}  (similaridade = {p['similaridade']})")
    print(f"    {p['id_1']}: {p['texto_1']}")
    print(f"    {p['id_2']}: {p['texto_2']}\n")

## 4. Matriz de Similaridade Completa (Heatmap)

In [ ]:
df_sim = pd.DataFrame(matriz_sim, index=ids, columns=ids)

plt.figure(figsize=(14, 12))
sns.heatmap(df_sim, cmap="Blues", vmin=0, vmax=1,
            square=True, cbar_kws={"label": "Similaridade de cosseno"})
plt.title(f"Matriz de Similaridade de Cosseno — {len(ids)} manifestações")
plt.tight_layout()
plt.savefig("matriz_similaridade.png", dpi=150)
plt.show()

## 5. Análise de Falsos Positivos e Falsos Negativos

O dataset sintético foi construído com um **gabarito conhecido** de duplicatas reais (ver `duplicatas_reais.json`), o que permite avaliar objetivamente a qualidade da detecção:

- **M003 × M017** — mesmo buraco/asfalto, descrito com palavras diferentes
- **M008 × M022** — mesma falta de médico, um usando o termo técnico "PSF"
- **M012 × M029** — mesmo descarte irregular de lixo em terreno baldio/vazio

In [ ]:
with open("duplicatas_reais.json", encoding="utf-8") as f:
    duplicatas_reais = [tuple(p) for p in json.load(f)]

duplicatas_reais_set = {frozenset(p) for p in duplicatas_reais}
pares_detectados_set = {frozenset((p["id_1"], p["id_2"])) for p in pares_encontrados}

verdadeiros_positivos = duplicatas_reais_set & pares_detectados_set
falsos_negativos = duplicatas_reais_set - pares_detectados_set
falsos_positivos = pares_detectados_set - duplicatas_reais_set

print(f"Duplicatas reais no dataset: {len(duplicatas_reais_set)}")
print(f"Verdadeiros positivos (corretamente detectados): {len(verdadeiros_positivos)}")
for p in verdadeiros_positivos:
    print("   ", tuple(p))
print(f"\nFalsos negativos (duplicatas reais NÃO detectadas): {len(falsos_negativos)}")
for p in falsos_negativos:
    print("   ", tuple(p))
print(f"\nFalsos positivos (detectados mas não são duplicatas reais): {len(falsos_positivos)}")
for p in falsos_positivos:
    print("   ", tuple(p))

precisao = len(verdadeiros_positivos) / len(pares_detectados_set) if pares_detectados_set else float("nan")
cobertura = len(verdadeiros_positivos) / len(duplicatas_reais_set)
print(f"\nPrecisão: {precisao:.2%}" if pares_detectados_set else "\nPrecisão: N/A (nenhum par detectado)")
print(f"Cobertura (recall): {cobertura:.2%}")

### Interpretação

Se a cobertura (recall) ficar abaixo de 100%, significa que o limiar de 0.85 é conservador demais para algum dos pares reais — vale reexecutar a célula acima com `LIMIAR = 0.75` ou `0.70` para verificar se esses pares passam a ser capturados, e observar quantos falsos positivos adicionais aparecem em troca. Esse é exatamente o trade-off precisão × cobertura que justifica a escolha de um limiar ligeiramente conservador para um sistema de triagem que futuramente pode alimentar decisões automatizadas.

## 6. Teste com limiar alternativo (sensibilidade)

In [ ]:
for limiar_teste in [0.70, 0.75, 0.80, 0.85, 0.90]:
    pares_t, _ = detectar_duplicatas(textos, limiar=limiar_teste, ids=ids, embeddings=embeddings)
    pares_t_set = {frozenset((p["id_1"], p["id_2"])) for p in pares_t}
    vp = len(duplicatas_reais_set & pares_t_set)
    fp = len(pares_t_set - duplicatas_reais_set)
    print(f"limiar={limiar_teste:.2f}  ->  {len(pares_t):3d} pares detectados "
          f"(verdadeiros positivos: {vp}/{len(duplicatas_reais_set)}, falsos positivos: {fp})")